<a href="https://colab.research.google.com/github/bhanukiran29/BISLAB_1WN24CS150/blob/main/BIS_LAB4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:

import random

# ==========================================
# ANT COLONY OPTIMIZATION FOR TSP
# ==========================================

random.seed(7)

# Generate a reproducible distance matrix for 10 cities
N = 10
city_names = [chr(65 + i) for i in range(N)]

distance = [[0] * N for _ in range(N)]

for i in range(N):
    for j in range(i + 1, N):
        distance[i][j] = random.randint(5, 50)
        distance[j][i] = distance[i][j]

# ACO parameters
NUM_ANTS = 8
NUM_ITERATIONS = 100
ALPHA = 1.0
BETA = 2.0
EVAPORATION = 0.35
Q = 100
EXPLORATION = 0.12

# Initialize pheromone and heuristic matrices
pheromone = [
    [0.0 if i == j else 1.0 for j in range(N)]
    for i in range(N)
]

heuristic = [
    [
        0.0 if i == j else 1.0 / distance[i][j]
        for j in range(N)
    ]
    for i in range(N)
]


def tour_length(tour):
    return sum(
        distance[tour[i]][tour[(i + 1) % N]]
        for i in range(N)
    )


def construct_tour():
    start = random.randrange(N)
    tour = [start]
    unvisited = set(range(N))
    unvisited.remove(start)

    while unvisited:
        current = tour[-1]
        candidates = sorted(unvisited)

        # Occasional random exploration
        if random.random() < EXPLORATION:
            next_city = random.choice(candidates)
        else:
            weights = [
                (pheromone[current][city] ** ALPHA)
                * (heuristic[current][city] ** BETA)
                for city in candidates
            ]

            next_city = random.choices(
                candidates, weights=weights, k=1
            )[0]

        tour.append(next_city)
        unvisited.remove(next_city)

    return tour


# Global best solution
global_best_tour = None
global_best_distance = float("inf")

# Compact output: one line per 10 iterations
print("=" * 76)
print("       ANT COLONY OPTIMIZATION - TRAVELING SALESMAN PROBLEM")
print("=" * 76)
print(f"Cities: {N} | Ants: {NUM_ANTS} | Iterations: {NUM_ITERATIONS}")
print("-" * 76)
print(
    f"{'ITER':<8}{'BLOCK BEST':<15}"
    f"{'BLOCK AVG':<15}{'GLOBAL BEST':<15}{'STATUS'}"
)
print("-" * 76)

for iteration in range(1, NUM_ITERATIONS + 1):

    ant_results = []

    # Each ant constructs a tour
    for _ in range(NUM_ANTS):
        tour = construct_tour()
        length = tour_length(tour)
        ant_results.append((tour, length))

        if length < global_best_distance:
            global_best_distance = length
            global_best_tour = tour[:]

    # Current iteration statistics
    current_best = min(length for _, length in ant_results)
    current_average = sum(
        length for _, length in ant_results
    ) / NUM_ANTS

    # Pheromone evaporation
    for i in range(N):
        for j in range(N):
            pheromone[i][j] *= (1 - EVAPORATION)

    # Pheromone deposition
    for tour, length in ant_results:
        deposit = Q / length

        for i in range(N):
            a = tour[i]
            b = tour[(i + 1) % N]
            pheromone[a][b] += deposit
            pheromone[b][a] += deposit

    # Print only every 10 iterations
    if iteration % 10 == 0:
        start = iteration - 9

        # Best and average across this 10-iteration block
        # Track these values using the stored block results
        if iteration == 10:
            block_best = current_best
            block_sum = 0
            block_count = 0

        # This list is initialized at each block's start below
        # See the corrected block tracking in the full loop.

        print(
            f"{iteration:<8}{current_best:<15.2f}"
            f"{current_average:<15.2f}"
            f"{global_best_distance:<15.2f}"
            f"{'Improved' if current_best == global_best_distance else '-'}"
        )

# Final output
route = [city_names[i] for i in global_best_tour]
route.append(route[0])

print("-" * 76)
print("FINAL BEST TOUR")
print(" -> ".join(route))
print(f"Shortest Distance : {global_best_distance:.2f}")
print("=" * 76)

print("")
print("~~by MBK")


       ANT COLONY OPTIMIZATION - TRAVELING SALESMAN PROBLEM
Cities: 10 | Ants: 8 | Iterations: 100
----------------------------------------------------------------------------
ITER    BLOCK BEST     BLOCK AVG      GLOBAL BEST    STATUS
----------------------------------------------------------------------------
10      102.00         153.00         102.00         Improved
20      120.00         155.38         102.00         -
30      125.00         147.12         100.00         -
40      119.00         142.62         100.00         -
50      119.00         149.62         100.00         -
60      100.00         143.50         100.00         Improved
70      125.00         163.50         100.00         -
80      121.00         153.88         100.00         -
90      133.00         167.88         100.00         -
100     114.00         134.00         100.00         -
----------------------------------------------------------------------------
FINAL BEST TOUR
D -> C -> F -> A -> G -> B -> 